# ⚙️ Modul 02 Hands-on: The Learning Engine & Optimization

Pada hands-on ini, Anda akan membongkar mesin penggerak neural network:
1. **Loss Function Behavior**: Membandingkan kurva penalti MSE vs Binary Cross-Entropy.
2. **The Optimizer Race**: Mengadu SGD, Momentum, dan Adam pada lanskap lembah curam untuk melihat siapa yang paling cepat konvergen.
3. **The Sabotage Lab**: Menyaksikan langsung apa yang terjadi saat Learning Rate meledak (*Loss NaN*) vs Learning Rate terlalu kecil (*Frozen Model*).
4. **Learning Rate Scheduling**: Menerapkan Cosine Annealing untuk menghaluskan konvergensi akhir.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Gunakan akselerasi Apple Silicon jika ada
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Device aktif: {device}")

## 1. Membedah Karakteristik Loss: MSE vs Cross-Entropy
Bayangkan label asli adalah $y = 1$ (kelas positif).
Kita variasikan tebakan model $\hat{y}$ dari $0.001$ sampai $0.999$, lalu lihat seberapa keras MSE vs Cross-Entropy memberi penalti.

In [ ]:
y_hat = torch.linspace(0.001, 0.999, 500)
y_true = 1.0

# 1. MSE: (1 - y_hat)^2
loss_mse = (y_true - y_hat) ** 2

# 2. Cross-Entropy: -log(y_hat)
loss_bce = -torch.log(y_hat)

plt.figure(figsize=(9, 5))
plt.plot(y_hat.numpy(), loss_mse.numpy(), label="Mean Squared Error (MSE)", linewidth=2.5, color="crimson")
plt.plot(y_hat.numpy(), loss_bce.numpy(), label="Binary Cross-Entropy (BCE)", linewidth=2.5, color="navy")
plt.title("Perbandingan Penalti Eror: Label Asli y = 1", fontsize=13, fontweight="bold")
plt.xlabel("Probabilitas Tebakan Model (y_hat)")
plt.ylabel("Nilai Penalti (Loss)")
plt.ylim(0, 6)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(fontsize=11)
plt.annotate("BCE meledakkan penalti\nke tak hingga jika tebakan salah!", xy=(0.02, 3.9), xytext=(0.15, 4.5),
             arrowprops=dict(facecolor="black", arrowstyle="->"), fontsize=10, fontweight="bold")
plt.show()

## 2. The Optimizer Race: SGD vs Momentum vs Adam
Kita buat lanskap fungsi 2D dengan lembah yang sangat curam (*elongated valley*):
$$f(x, y) = 0.1 x^2 + 2.0 y^2$$
Titik minimum berada di $(0, 0)$. Semua optimizer mulai dari titik yang sama $(-4.0, 3.0)$. Mari kita amati jejak jalannya!

In [ ]:
def loss_func(x, y):
    return 0.1 * x**2 + 2.0 * y**2

def run_optimizer(opt_class, opt_kwargs, steps=50):
    p = torch.tensor([-4.0, 3.0], requires_grad=True)
    optimizer = opt_class([p], **opt_kwargs)
    trajectory = [p.detach().clone().numpy()]
    
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_func(p[0], p[1])
        loss.backward()
        optimizer.step()
        trajectory.append(p.detach().clone().numpy())
        
    return np.array(trajectory)

# Jalankan ketiga optimizer dengan learning rate yang seimbang
traj_sgd = run_optimizer(optim.SGD, {"lr": 0.4})
traj_mom = run_optimizer(optim.SGD, {"lr": 0.4, "momentum": 0.8})
traj_adam = run_optimizer(optim.Adam, {"lr": 0.4})

# Plot Kontur Lanskap dan Jejak Optimizer
x_vals = np.linspace(-5, 5, 200)
y_vals = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x_vals, y_vals)
Z = 0.1 * X**2 + 2.0 * Y**2

plt.figure(figsize=(10, 6))
plt.contour(X, Y, Z, levels=30, cmap="viridis", alpha=0.6)
plt.plot(traj_sgd[:, 0], traj_sgd[:, 1], "o-", label="Vanilla SGD (Zigzag & Lambat)", color="red", markersize=4)
plt.plot(traj_mom[:, 0], traj_mom[:, 1], "s-", label="SGD + Momentum (Inersia Tinggi)", color="blue", markersize=4)
plt.plot(traj_adam[:, 0], traj_adam[:, 1], "^--", label="Adam (Adaptif & Cepat ke Pusat!)", color="green", markersize=5)
plt.plot(0, 0, "r*", markersize=15, label="Titik Minimum Global (0,0)")
plt.title("The Optimizer Race: Menuruni Lembah Curam (50 Steps)", fontsize=13, fontweight="bold")
plt.xlabel("Parameter x")
plt.ylabel("Parameter y")
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 3. The Sabotage Lab: Malapetaka Learning Rate
Learning Rate adalah hyperparameter paling krusial. Apa yang terjadi jika salah mengaturnya?
* **Eksperimen 1**: $lr = 10.0$ (Terlalu Besar $\rightarrow$ Gradien meledak melompati lembah, output menjadi `NaN`).
* **Eksperimen 2**: $lr = 10^{-6}$ (Terlalu Kecil $\rightarrow$ Model beku, loss hampir tidak berubah).
* **Eksperimen 3**: $lr = 0.01$ (Tepat $\rightarrow$ Konvergen mulus).

In [ ]:
# Kita uji pada data klasifikasi sintetis sederhana
X_toy = torch.randn(200, 10).to(device)
y_toy = torch.randint(0, 2, (200, 1)).float().to(device)

def train_with_lr(lr_value, name):
    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr_value)
    
    loss_history = []
    for epoch in range(1, 41):
        optimizer.zero_grad()
        out = model(X_toy)
        loss = criterion(out, y_toy)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())
        
    return loss_history

loss_too_large = train_with_lr(10.0, "LR = 10.0")
loss_too_small = train_with_lr(1e-5, "LR = 1e-5")
loss_just_right = train_with_lr(0.1, "LR = 0.1")

plt.figure(figsize=(10, 5))
plt.plot(loss_too_large, "r--", label=f"LR Terlalu Besar (10.0) -> Akhir: {loss_too_large[-1]:.4f}", linewidth=2)
plt.plot(loss_too_small, "b-", label=f"LR Terlalu Kecil (1e-5) -> Akhir: {loss_too_small[-1]:.4f}", linewidth=2)
plt.plot(loss_just_right, "g-", label=f"LR Tepat (0.1) -> Akhir: {loss_just_right[-1]:.4f}", linewidth=2.5)
plt.title("Sabotage Lab: Dampak Learning Rate terhadap Kurva Loss", fontsize=13, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 4. Learning Rate Scheduler: Cosine Annealing in Action
Bagaimana cara profesional agar training cepat di awal tapi stabil di akhir?
Gunakan `torch.optim.lr_scheduler.CosineAnnealingLR`.

In [ ]:
dummy_param = [torch.nn.Parameter(torch.randn(2, 2))]
optimizer = optim.AdamW(dummy_param, lr=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-5)

lrs = []
for epoch in range(100):
    lrs.append(scheduler.get_last_lr()[0])
    scheduler.step()

plt.figure(figsize=(8, 4))
plt.plot(lrs, color="purple", linewidth=2.5)
plt.title("Jadwal Penurunan Learning Rate (Cosine Annealing)", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 5. 🏆 Hands-on Challenge: Melatih Model dengan AdamW & Cosine Annealing

Lengkapi cell di bawah ini untuk melatih model klasifikasi biner 3-layer pada data `X_toy` menggunakan:
1. Optimizer: **AdamW** dengan `lr=0.02`
2. Scheduler: **CosineAnnealingLR** (`T_max=50`)
3. Cetak loss di epoch 1, 25, dan 50!

In [ ]:
# TODO: Selesaikan kode di bawah ini
model_challenge = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
).to(device)

criterion = nn.BCEWithLogitsLoss()
opt = optim.AdamW(model_challenge.parameters(), lr=0.02, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50, eta_min=1e-4)

print("Mulai Training Challenge...")
for epoch in range(1, 51):
    opt.zero_grad()
    preds = model_challenge(X_toy)
    loss = criterion(preds, y_toy)
    loss.backward()
    opt.step()
    sched.step()
    
    if epoch in [1, 25, 50]:
        current_lr = sched.get_last_lr()[0]
        print(f"Epoch {epoch:2d}/50 | Loss: {loss.item():.4f} | Current LR: {current_lr:.6f}")

print("\n🎉 Selamat! Anda telah menguasai mekanisme Loss, Optimizer, dan LR Scheduling!")